# Data Reading and Preprocessing


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import timeit
from collections import defaultdict
from contextlib import redirect_stdout
from io import StringIO
from itertools import combinations

import pandas as pd
from scipy.stats import trim_mean

In [7]:
df = pd.read_csv(os.path.join("dataset", "Market_Basket_Optimisation_binarized.csv"))
print(df.info())
df.head(3)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7501 entries, 0 to 7500
Columns: 119 entries, almonds to zucchini
dtypes: int64(119)
memory usage: 6.8 MB
None


,almonds,antioxydant juice,asparagus,avocado,babies food,bacon,barbecue sauce,black tea,blueberries,body spray,...,turkey,vegetables mix,water spray,white wine,whole weat flour,whole wheat pasta,whole wheat rice,yams,yogurt cake,zucchini
0,1,1,0,1,0,0,0,0,0,0,...,0,1,0,0,1,0,0,1,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


**get_support** function evaluates the support value for a set given all the transactions.


In [8]:
def get_support(df: pd.DataFrame, itemset: set) -> float:

    if len(df) == 0:
        return 0.0

    return df.iloc[:, sorted(itemset)].all(axis=1).mean()



def test_get_support():
    # Create a DataFrame
    _df = pd.DataFrame(
        {
            "A": [1, 1, 0, 0, 1],
            "B": [1, 0, 1, 1, 0],
            "C": [0, 1, 0, 1, 1],
        }
    )

    # Test the function
    assert get_support(_df, {0}) == 0.6
    assert get_support(_df, {1}) == 0.6
    assert get_support(_df, {2}) == 0.6
    assert get_support(_df, {0, 1}) == 0.2
    assert get_support(_df, {0, 2}) == 0.4
    assert get_support(_df, {1, 2}) == 0.2
    assert get_support(_df, {0, 1, 2}) == 0.0

    print("All test cases passed!.")


# Run the test cases
test_get_support()

All test cases passed!.


In [9]:
def get_subsets(item_set):
    return [item_set - {item} for item in item_set]


def test_get_subsets():
    # Test case 1: Empty item set
    item_set_empty = set()
    expected_output_empty = []
    assert get_subsets(item_set_empty) == expected_output_empty

    # Test case 2: Non-empty item set
    item_set_non_empty = {1, 2, 3}
    expected_output_non_empty = [{2, 3}, {1, 3}, {1, 2}]
    assert get_subsets(
        item_set_non_empty) == expected_output_non_empty

    print("Test cases for get_subsets passed!")


test_get_subsets()

Test cases for get_subsets passed!


In [10]:
def is_valid_set(item_set, prev_level_sets):
    """ 
    Check if all the subsets of the item_set are present in the previous level sets.
    
    Parameters:
        item_set (list): The item set to be validated.
        prev_level_sets (list of lists): List of sets from the previous level.
        
    Returns:
        bool: True if all subsets of the item_set are present in prev_level_sets, False otherwise.
      
    Example:
        >>> prev_level_sets = [{1, 2}, {2, 3}, {1, 3}]
        >>> item_set = {1, 2, 4}
        in this example, a subset of item_set {1, 4} is not present in prev_level_sets.
        following the principle that if a set is not frequent, all its supersets are also not frequent,
        and since {1, 4} is not frequent, ie: not present in prev_level_sets, {1, 2, 4} is also not frequent.
    """
    if len(prev_level_sets) == 0:
        return False
    
    single_drop_subsets = get_subsets(item_set)
    for single_drop_set in single_drop_subsets:
        if single_drop_set not in prev_level_sets:
            return False
        
    return True


def test_is_valid_set():
    # Test case 1: Empty previous level sets
    prev_level_sets_empty = []
    item_set = {1, 2}
    assert is_valid_set(item_set, prev_level_sets_empty) is False

    # Test case 2: Item set not present in previous level sets
    prev_level_sets = [{3, 4}, {5, 6}]
    item_set_not_present = {1, 2}
    assert is_valid_set(item_set_not_present, prev_level_sets) is False

    # Test case 3: Item set present in previous level sets
    prev_level_sets_present = [{1, 2}, {2, 3}, {1, 3}, {3, 4}, {5, 6}]
    item_set_present = {1, 2, 3}
    assert is_valid_set(item_set_present, prev_level_sets_present) is True

    print("Test cases for is_valid_set passed!")


# Run the test cases
test_is_valid_set()

Test cases for is_valid_set passed!


**generate_candidate_item_sets** function generates the candidate item sets of size k from the frequent item sets of size k-1.


In [11]:
def generate_candidate_item_sets(frequent_item_sets, level):
    current_level_candidates = list()

    if len(frequent_item_sets[level - 1]) == 0:
        return current_level_candidates

    # Extract unique items from the frequent item sets of the previous level
    unique_items = set()
    prev_level_sets = list()
    for item_set, _ in frequent_item_sets[level - 1]:
        unique_items.update(item_set)
        prev_level_sets.append(item_set)

    # Generate candidates by combining unique items
    for candidate_set in combinations(unique_items, level + 1):
        # convert the tuple to a set
        candidate_set = set(candidate_set)
        # if the candidate set has a subset that doesn't exist in the frequent item sets of the previous level, skip it
        if is_valid_set(candidate_set, prev_level_sets):
            current_level_candidates.append(candidate_set)

    # calculate the support of each candidate set
    return current_level_candidates

**prune_candidates** function prunes the candidate sets evaluated based on the mean of the support for the current level.

In [ ]:
def prune_candidates(df: pd.DataFrame, current_level_candidates, trim_fraction=0.1):
    post_prune_candidates_set = list()

    supports = []
    for candidate_set in current_level_candidates:
        support = get_support(df, candidate_set)
        supports.append(support)
        post_prune_candidates_set.append((candidate_set, support))

    trimmed_mean_support = trim_mean(supports, proportiontocut=trim_fraction)

    return [
        (candidate_set, support)
        for candidate_set, support in post_prune_candidates_set
        if support >= trimmed_mean_support
    ], trimmed_mean_support


def test_prune_candidates():
    # Mock dataset
    data = {
        "A": [1, 1, 0, 0],
        "B": [1, 0, 1, 0],
        "C": [0, 1, 1, 0],
        "D": [1, 1, 1, 1],
    }
    df = pd.DataFrame(data)

    # Test case 1: Empty candidate set
    current_level_candidates = []
    expected_output_empty = []
    assert prune_candidates(df, current_level_candidates)[0] == expected_output_empty

    # Test case 2: Candidate set with valid itemsets
    current_level_candidates = [{0, 1}, {1, 2}, {2, 3}]
    
    expected_output_valid = [
        ({2, 3}, 0.5)
    ]
    assert prune_candidates(df, current_level_candidates)[0] == expected_output_valid

    print("All test cases passed!")


test_prune_candidates()

All test cases passed!


## Main apriori algorithm


In [ ]:
def apriori(df: pd.DataFrame, trimmed_mean_fraction=0.1):
    frequent_item_sets = defaultdict(list)
    history = pd.DataFrame(columns=["Level", "Candidates", "Frequent Itemsets", "Min Support"])

    means = df.mean()
    min_support = trim_mean(means, proportiontocut=trimmed_mean_fraction)

    for i, support in enumerate(means[means >= min_support]):
        frequent_item_sets[0].append(({i}, support))

    # Add the first row using loc
    history.loc[len(history)] = [0, len(df.columns), len(frequent_item_sets[0]), min_support]

    total_itemsets = 0
    for level in range(1, len(df.columns)):

        current_level_candidates = generate_candidate_item_sets(frequent_item_sets, level)

        if len(current_level_candidates) == 0:
            break

        frequent_item_sets[level], trimmed_mean_support = prune_candidates(
            df, current_level_candidates, trimmed_mean_fraction
        )
        num_frequent_itemsets = len(frequent_item_sets[level])
        total_itemsets += num_frequent_itemsets

        # Add each level's data to history using loc
        history.loc[len(history)] = [level, len(current_level_candidates), num_frequent_itemsets, trimmed_mean_support]

    return frequent_item_sets, history


In [48]:
trimmed_mean = 0.1
frequent_item_sets, history = apriori(df, trimmed_mean)
history

,Level,Candidates,Frequent Itemsets,Min Support
0,0.0,119.0,47.0,0.023876
1,1.0,1081.0,415.0,0.000622
2,2.0,2289.0,1093.0,0.000223
3,3.0,1403.0,411.0,0.000138
4,4.0,69.0,50.0,0.000236
5,5.0,2.0,2.0,0.000267


In [ ]:
num_iterations = 2

with redirect_stdout(StringIO()):
    time_taken = timeit.timeit(lambda: apriori(df), number=num_iterations)

print(f"Time taken for {num_iterations} iterations of apriori: {time_taken} seconds")

Time taken for 2 iterations of apriori: 27.94829629996093 seconds


## Generating Association Rules

In [28]:
item_support_dict = dict()

item_list = list()
for level in frequent_item_sets:
    for set_support_pair in frequent_item_sets[level]:
        for i in set_support_pair[0]:
            item_list.append(df.columns[i])
        item_support_dict[frozenset(item_list)] = set_support_pair[1]
        item_list = list()

In [29]:
item_support_dict

{frozenset({'almonds'}): 0.03332888948140248,
 frozenset({'antioxydant juice'}): 0.03372883615517931,
 frozenset({'asparagus'}): 0.0871883748833489,
 frozenset({'avocado'}): 0.030129316091187842,
 frozenset({'babies food'}): 0.08105585921877083,
 frozenset({'bacon'}): 0.025729902679642713,
 frozenset({'barbecue sauce'}): 0.04679376083188908,
 frozenset({'black tea'}): 0.05999200106652446,
 frozenset({'blueberries'}): 0.1638448206905746,
 frozenset({'body spray'}): 0.08038928142914278,
 frozenset({'bramble'}): 0.0510598586855086,
 frozenset({'brownies'}): 0.03186241834422077,
 frozenset({'bug spray'}): 0.17970937208372217,
 frozenset({'burger sauce'}): 0.027063058258898813,
 frozenset({'burgers'}): 0.026663111585121985,
 frozenset({'butter'}): 0.0793227569657379,
 frozenset({'cake'}): 0.1709105452606319,
 frozenset({'candy bars'}): 0.043060925209972005,
 frozenset({'carrots'}): 0.06332489001466471,
 frozenset({'cauliflower'}): 0.09532062391681109,
 frozenset({'cereals'}): 0.052393014264

## association_rules 
generates the association rules in accordance with the given _minimum confidence_ value and the provided dictionary of itemsets against their support values. For itemsets of more than one element, it first finds all their subsets. For every subset A, it calculates the set B = itemset-A. If B is not empty, the confidence of B is calculated. If this value is more than _minimum confidence_ value, the rule _A->B_ is added to the list.


In [326]:
def association_rules(min_confidence, min_lift, support_dict) -> pd.DataFrame:
    rules_data = []

    for itemset, support in support_dict.items():

        if len(itemset) < 2:
            continue

        for i in range(1, len(itemset)):
            for A in combinations(itemset, i):
                A = frozenset(A)
                B = itemset - A

                support_A = support_dict[A]
                support_B = support_dict[B]

                confidence = support / support_A
                lift = support / (support_A * support_B)

                if confidence >= min_confidence and lift >= min_lift:
                    rules_data.append(
                        {"A": A, "B": B, "Confidence": confidence, "Lift": lift}
                    )

    rules_df = pd.DataFrame(rules_data)
    return rules_df

### Specify a minimum confidence and lift value to generate the association rules.

In [334]:
min_confidence = 0.5
min_lift = 0
rules = association_rules(
    min_confidence=min_confidence,
    min_lift=min_lift,
    support_dict=item_support_dict,
)

In [336]:
import timeit
# benchmarking the association_rules function
time_taken_association_rules = timeit.timeit(
    lambda: association_rules(min_confidence, min_lift, item_support_dict),
    number=num_iterations,
)

print(
    f"Time taken for {num_iterations} iterations of association_rules: {time_taken_association_rules} seconds"
)

Time taken for 100 iterations of association_rules: 0.14365209999959916 seconds


In [328]:
print("Number of rules: ", len(rules), "\n")
rules

Number of rules:  53 



,A,B,Confidence,Lift
0,"(almonds, carrots)",(burgers),0.571429,5.994805
1,"(almonds, chicken)",(burgers),0.555556,5.828283
2,"(almonds, cooking oil)",(chocolate),0.538462,5.664797
3,"(burgers, bug spray)",(chocolate),0.500000,5.260168
4,"(cake, cider)",(burgers),0.545455,5.722314
5,"(cooking oil, chili)",(chicken),0.714286,2.996564
6,"(chili, chicken)",(cooking oil),0.555556,8.885335
7,"(almonds, chicken, chocolate)",(burgers),0.500000,5.245455
8,"(cereals, almonds, chocolate)",(cake),0.666667,5.046081
9,"(cereals, almonds, cake)",(chocolate),0.666667,7.013558
